In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/16class-pretrained/pytorch/default/1/16class-transformers-default-v1/__huggingface_repos__.json
/kaggle/input/16class-pretrained/pytorch/default/1/16class-transformers-default-v1/modelout/config.json
/kaggle/input/16class-pretrained/pytorch/default/1/16class-transformers-default-v1/modelout/merges.txt
/kaggle/input/16class-pretrained/pytorch/default/1/16class-transformers-default-v1/modelout/vocab.json
/kaggle/input/16class-pretrained/pytorch/default/1/16class-transformers-default-v1/modelout/tokenizer_config.json
/kaggle/input/16class-pretrained/pytorch/default/1/16class-transformers-default-v1/modelout/model.safetensors
/kaggle/input/16class-pretrained/pytorch/default/1/16class-transformers-default-v1/modelout/special_tokens_map.json


In [2]:
!pip install nest_asyncio uvicorn fastapi pydantic pyngrok python-multipart


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.4/66.4 kB 856.3 kB/s eta 0:00:000:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.6/95.6 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.0/73.0 kB 3.5 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import torch
from fastapi import FastAPI, HTTPException, UploadFile, File
from fastapi.responses import StreamingResponse
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.utils.data import Dataset, DataLoader
from pyngrok import ngrok
import uvicorn
import io
import re
import logging
import threading
import nest_asyncio

# Apply nest_asyncio for environments with existing event loops
nest_asyncio.apply()

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# Initialize FastAPI app
app = FastAPI()

# Define batch size
BATCH_SIZE = 32

# Initialize models and tokenizers
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
logger.info("Loading classification models...")

sentiment_model_name = "j-hartmann/sentiment-roberta-large-english-3-classes"
sentiment_tokenizer = AutoTokenizer.from_pretrained(sentiment_model_name)
sentiment_model = AutoModelForSequenceClassification.from_pretrained(sentiment_model_name).to(device)

class16_model_path = "/kaggle/input/16class-pretrained/pytorch/default/1/16class-transformers-default-v1/modelout"
class16_tokenizer = AutoTokenizer.from_pretrained(class16_model_path)
class16_model = AutoModelForSequenceClassification.from_pretrained(class16_model_path).to(device)

logger.info("Models loaded successfully")

# Define label map for 16-class classifier
label_map = {
    "Accessibility": 0, "App Performance": 1, "Bans & Restrictions": 2, "Company Policies": 3,
    "Customer Support": 4, "Discounts & Offers": 5, "Driver Experience": 6, "Payment & Transactions": 7,
    "Pricing": 8, "Ratings & Reviews": 9, "Regulations & Legal": 10, "Ride Availability": 11,
    "Security": 12, "Sustainability & Environment": 13, "Trust & Safety": 14, "User Interface": 15
}
reverse_label_map = {v: k for k, v in label_map.items()}

# Custom Dataset class
class TextDataset(Dataset):
    def __init__(self, texts, tokenizer, max_length=384):
        self.texts = texts
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        encoding = self.tokenizer(
            text,
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0)
        }

def preprocess_text(text):
    """Standard text preprocessing for NLP tasks"""
    text = str(text) if pd.notna(text) else ""
    text = re.sub(r'http\S+|www\S+', '', text)  # Remove URLs
    text = re.sub(r'[^a-zA-Z\s.,!?]', '', text)  # Keep letters and basic punctuation
    text = text.lower()  # Convert to lowercase
    text = ' '.join(text.split())  # Normalize whitespace
    return text

@app.post("/analyze_csv")
async def analyze_csv(file: UploadFile = File(...)):
    """Analyze CSV file for sentiment and 16-class classification using Dataset"""
    logger.info(f"Analyzing file: {file.filename}")
    
    try:
        # Read CSV file
        content = await file.read()
        df = pd.read_csv(io.BytesIO(content))
        
        # Preprocess text data
        df['text'] = df['text'].apply(preprocess_text)
        texts = df['text'].tolist()
        ids = df.get('id', [f"item_{i}" for i in range(len(df))]).tolist()
        review_ids = df.get('review_id', [None]*len(df)).tolist()

        # Create datasets
        sentiment_dataset = TextDataset(texts, sentiment_tokenizer)
        class16_dataset = TextDataset(texts, class16_tokenizer)

        # Create data loaders
        sentiment_loader = DataLoader(sentiment_dataset, batch_size=BATCH_SIZE, shuffle=False)
        class16_loader = DataLoader(class16_dataset, batch_size=BATCH_SIZE, shuffle=False)

        # Set models to evaluation mode
        sentiment_model.eval()
        class16_model.eval()

        # Process batches
        results = []
        with torch.no_grad():
            for (sentiment_batch, class16_batch), batch_ids, batch_review_ids in zip(
                zip(sentiment_loader, class16_loader), 
                [ids[i:i + BATCH_SIZE] for i in range(0, len(ids), BATCH_SIZE)],
                [review_ids[i:i + BATCH_SIZE] for i in range(0, len(review_ids), BATCH_SIZE)]
            ):
                # Move batches to device
                sentiment_inputs = {k: v.to(device) for k, v in sentiment_batch.items()}
                class16_inputs = {k: v.to(device) for k, v in class16_batch.items()}

                # Get predictions
                sentiment_outputs = sentiment_model(**sentiment_inputs)
                class16_outputs = class16_model(**class16_inputs)

                # Process sentiment predictions
                sentiment_probs = torch.nn.functional.softmax(sentiment_outputs.logits, dim=-1)
                sentiment_labels = sentiment_probs.argmax(dim=-1)
                sentiment_confidences = sentiment_probs.max(dim=-1).values

                # Process 16-class predictions
                class16_probs = torch.nn.functional.softmax(class16_outputs.logits, dim=-1)
                class16_labels = class16_probs.argmax(dim=-1)
                class16_confidences = class16_probs.max(dim=-1).values

                # Compile results
                for i in range(len(batch_ids)):
                    sentiment_label = sentiment_model.config.id2label[sentiment_labels[i].item()]
                    if sentiment_label.startswith("LABEL_"):
                        sentiment_label = sentiment_label[6:]

                    class16_label_idx = class16_labels[i].item()
                    category = reverse_label_map.get(class16_label_idx, f"Unknown_{class16_label_idx}")

                    results.append({
                        "id": batch_ids[i],
                        "review_id": batch_review_ids[i],
                        "text": texts[i][:200],  # Use original text index
                        "sentiment": sentiment_label.lower(),
                        "sentiment_confidence": float(sentiment_confidences[i]),
                        "category": category,
                        "category_confidence": float(class16_confidences[i])
                    })

        # Generate CSV response
        output = io.StringIO()
        pd.DataFrame(results).to_csv(output, index=False)
        
        return StreamingResponse(
            iter([output.getvalue()]),
            media_type="text/csv",
            headers={"Content-Disposition": "attachment; filename=results.csv"}
        )

    except Exception as e:
        logger.error(f"Error processing CSV: {str(e)}")
        raise HTTPException(status_code=500, detail=str(e))

def start_ngrok():
    """Start ngrok tunnel in a separate thread"""
    try:
        ngrok.set_auth_token("2vBCiODgw3JDJbKW0DGDCrwypEJ_2qg1ttzKGXzamYTGxyMui")
        tunnel = ngrok.connect(8000, "http")
        public_url = tunnel.public_url
        logger.info(f"Ngrok tunnel established at: {public_url}")
        print(f"Public URL: {public_url}")
        return public_url
    except Exception as e:
        logger.error(f"Error starting ngrok: {str(e)}")
        raise

def start_server():
    """Start FastAPI server with ngrok"""
    ngrok_thread = threading.Thread(target=start_ngrok)
    ngrok_thread.start()
    uvicorn.run(app, host="0.0.0.0", port=8000)

if __name__ == "__main__":
    start_server()

tokenizer_config.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/725 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.43G [00:00<?, ?B/s]

Some weights of the model checkpoint at j-hartmann/sentiment-roberta-large-english-3-classes were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


INFO:     Started server process [31]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


Public URL: https://52044301f910.ngrok-free.app                                                     
INFO:     223.236.125.113:0 - "POST /analyze_csv HTTP/1.1" 200 OK
INFO:     223.236.125.113:0 - "POST /analyze_csv HTTP/1.1" 200 OK
INFO:     223.236.125.113:0 - "POST /analyze_csv HTTP/1.1" 200 OK
INFO:     223.236.125.113:0 - "POST /analyze_csv HTTP/1.1" 200 OK


In [ ]:
import torch
class16_model_path = "/kaggle/input/16class/transformers/default/1/modelout"
save_path = "/kaggle/working/modelout.pth"
torch.save(class16_model_path.state_dict(), save_path)